# Demo 09e — Separating a mixture, three ways

**Class 09 · E1** (slides: `slides/09_blind_signal_separation.md`)

The running example of the class: a 3 Hz sine and a 7 Hz chirp mixed by a 2 x 2 matrix.
Three decompositions, three views of the same cloud: PCA (variance, uncorrelated),
ICA (non-Gaussianity, independent), NMF (positivity, additive). One table, one conclusion.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pynnmf
from sklearn.decomposition import PCA, FastICA

from faa import bss, plotlib

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(20)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
pl.Config.set_tbl_width_chars(150)
plotlib.setup(module="09")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
SEED = 0


def slide(title: str) -> None:
    """Every cell that prints a number of a slide starts with this line."""
    print(f"--- slide: {title} ---")

## Slide: The mixture, the three methods, one table

PCA and ICA recover the sources by unmixing the rows; NMF needs non-negative data, so
the mixtures are shifted back to positive and the recovered components are matched.
`bss.match` pairs each estimate with its true source and fixes the sign.

In [ ]:
X, S, A = bss.mixtures(n=300, seed=SEED)
pca = PCA(2).fit(X.T)
S_pca = bss.match(S, pca.transform(X.T).T)[0]  # scores: the projected sensor data, one row per component
ica = FastICA(2, random_state=SEED).fit(X.T)
S_ica = bss.match(S, ica.transform(X.T).T)[0]
rec, W, H, _ = pynnmf.nmf_mu(X - X.min() + 0.05, 2, n=500, seed=SEED)  # shift: NMF needs X >= 0
S_nmf = bss.match(S, H)[0]
slide("Three separations of one 2-sensor mixture")
rows = []
for name, est in [("PCA", S_pca), ("ICA", S_ica), ("NMF (shifted)", S_nmf)]:
    for i in range(2):
        c = max(abs(np.corrcoef(est[j], S[i])[0, 1]) for j in range(2))  # match blind: best of the two estimates
        rows.append(
            {
                "method": name,
                "source": f"s{i + 1}",
                "corr": c,
                "SNR (dB)": 10 * np.log10(max(c**2, 1e-12) / (1 - min(c**2, 1 - 1e-12))),
            }
        )
print(
    pl.DataFrame(rows)
    .with_columns(pl.col("corr").round(3), pl.col("SNR (dB)").round(1))
    .pivot(values=["corr", "SNR (dB)"], on="source", separator=" ", index="method")
)

## Slide: What each method assumes — the constraint map

The same scatter plot, three views: PCA finds the axes of variance, ICA the axes of
independence (kurtosis), NMF the positive cone. The figure is the argument.

In [ ]:
f, ax = plt.subplots(1, 3, figsize=(plotlib.FULL, 2.4))
ax[0].scatter(X[0], X[1], s=3, color=GREY)
for v in pca.components_:
    ax[0].axline((0, 0), slope=v[1] / v[0], color=BLUE, lw=1)
ax[0].set_title("PCA directions", fontsize=8)
ax[1].scatter(X[0], X[1], s=3, color=GREY)
for v in ica.components_:
    ax[1].axline((0, 0), slope=v[1] / v[0], color=ORANGE, lw=1)
ax[1].set_title("ICA directions", fontsize=8)
ax[2].scatter(X[0] - X.min() + 0.05, X[1] - X.min() + 0.05, s=3, color=GREY)
ax[2].set_title("NMF parts (shifted)", fontsize=8)
for a in ax:
    a.set_xlabel("sensor 1", fontsize=7), a.set_ylabel("sensor 2", fontsize=7)
plotlib.save(f, "p5_three_views")
print("PCA directions are orthogonal; ICA directions are not; NMF has no directions, only additive parts.")

## Slide: More sensors help — 1, 2 and 3 mixtures of the same two sources

With one sensor the problem is ill-posed for these linear models (the second source
cannot be recovered). With three, the extra dimension absorbs noise.

In [ ]:
rows = []
for m in [1, 2, 3]:
    A_m = A[:1] if m == 1 else np.vstack([A, [0.5, 0.9]])  # 1 sensor: the first row; 3: an extra microphone
    if m == 1:
        rows.append({"sensors": m, "SNR s1 (dB)": float("nan"), "SNR s2 (dB)": float("nan")})
        continue
    Xm = A_m[:m] @ S + 0.05 * np.random.default_rng(SEED).normal(0, 1, (m, 300))
    ica_m = FastICA(m, random_state=SEED).fit(Xm.T)
    est = ica_m.transform(Xm.T).T  # (m, 300) recovered signals
    cs = [max(abs(np.corrcoef(est[j], S[i])[0, 1]) for j in range(m)) for i in range(2)]
    rows.append(
        {
            "sensors": m,
            "SNR s1 (dB)": 10 * np.log10(cs[0] ** 2 / (1 - cs[0] ** 2)),
            "SNR s2 (dB)": 10 * np.log10(cs[1] ** 2 / (1 - cs[1] ** 2)),
        }
    )
slide("Sensors sweep (ICA, two sources)")
print(pl.DataFrame(rows).with_columns(pl.col(pl.Float64).round(1)))

## Sweep and failure cases (more than the slides)

1. Gaussian sources: ICA has nothing to maximise — any orthogonal rotation is equally good.
2. NMF with the shift: the recovered parts include the offset; correlation is high but
   the amplitude is not the source's.

In [ ]:
rng = np.random.default_rng(SEED)
G = rng.normal(0, 1, (2, 300))
A_g = np.array([[0.9, 0.4], [0.7, 1.1]])
Xg = A_g @ G + 0.05 * rng.normal(0, 1, (2, 300))
ica_g = FastICA(2, random_state=SEED).fit(Xg.T)
est_g = ica_g.transform(Xg.T).T
slide("Failure: Gaussian sources")
print(f"kurtosis of the true sources: {bss.kurtosis(G).round(2)} (both ~ 0)")
print(f"ICA 'recovery' correlation:  {np.corrcoef(est_g[0], G[0])[0, 1]:.3f}, {np.corrcoef(est_g[1], G[1])[0, 1]:.3f}")
print("high correlation is luck: with Gaussian sources every rotation is a solution")

In [ ]:
slide("Failure: NMF amplitude after the shift")
print(f"true source std:  {S.std(axis=1).round(2)}")
print(f"NMF part std:     {(S_nmf - S_nmf.mean(1, keepdims=True)).std(axis=1).round(2)} (scale is arbitrary anyway)")